# BirdCLEF+ 2026 Perch v2 Probe

Extract frozen Google Perch v2 embeddings and train a shallow PyTorch probe. This notebook measures the value of foundation-model acoustic features for BirdCLEF+ 2026.


## 1. Environment


In [1]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")
    tensorflow_wheel_root = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0")
    perch_input_roots = [
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2"),
        Path("/kaggle/input/perch-meta"),
    ]


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def package_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def install_tensorflow_220() -> None:
    tf_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorflow-2.20.0*.whl"))
    tb_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorboard-2.20.0*.whl"))
    if not tf_wheels:
        raise FileNotFoundError(
            f"TensorFlow 2.20 wheel not found under {CFG.tensorflow_wheel_root}. "
            "Attach the bc26-tensorflow-2-20-0 Kaggle input before running notebook 3."
        )
    targets = [str(path) for path in tb_wheels[:1] + tf_wheels[:1]]
    print(f"Installing TensorFlow 2.20 from {CFG.tensorflow_wheel_root}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *targets], check=True)


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

import librosa
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple("2.20.0"):
    print(f"TensorFlow {installed_tf} is older than the Perch v2 requirement; installing 2.20.0 from Kaggle input.")
    install_tensorflow_220()

try:
    import tensorflow as tf
except ImportError:
    tf = None

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    sample_rate = 32000
    duration = 5.0
    embedding_dim = 1536
    extraction_batch_size = 8
    probe_batch_size = 128
    probe_epochs = 20
    early_stopping_patience = 4
    early_stopping_min_delta = 1e-4
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    perch_model_dir = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if tf is not None:
    print(f"TensorFlow: {tf.__version__}")


Data root: /kaggle/input/competitions/birdclef-2026
Output directory: /kaggle/working/artifacts
TensorFlow 2.19.0 is older than the Perch v2 requirement; installing 2.20.0 from Kaggle input.
Installing TensorFlow 2.20 from /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0
Device: cuda
TensorFlow: 2.20.0


## 2. Metadata


Use the same 206 primary labels as the EfficientNet baseline so the comparison isolates representation quality rather than target design.


In [2]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if CFG.max_samples:
    train = train.sample(CFG.max_samples, random_state=CFG.seed).reset_index(drop=True)

labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)
(CFG.artifact_dir / "labels.json").write_text(json.dumps(idx_to_label, indent=2), encoding="utf-8")

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
display(train.head())


Rows: 35,549
Classes: 206


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,filepath,target
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0


## 3. Load Perch v2


Load the TensorFlow SavedModel and run a small smoke test before extraction. This catches TensorFlow/XLA compatibility issues before the expensive audio loop.


In [3]:
def find_perch_model_dir() -> Path:
    if CFG.perch_model_dir:
        path = Path(CFG.perch_model_dir)
        if (path / "saved_model.pb").exists():
            return path
        matches = list(path.glob("**/saved_model.pb")) if path.exists() else []
        if matches:
            return matches[0].parent

    search_roots = [path for path in CFG.perch_input_roots if path.exists()]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        search_roots.append(input_root)

    matches = []
    for root in dict.fromkeys(search_roots):
        if (root / "saved_model.pb").exists():
            matches.append(root / "saved_model.pb")
        matches.extend(root.glob("**/saved_model.pb"))
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower() or "bird" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find a Perch SavedModel. Attach /kaggle/input/datasets/jaejohn/perch-meta, "
        "the Perch/vocalization-classifier Kaggle model, or set CFG.perch_model_dir."
    )


if tf is None:
    raise ImportError("TensorFlow is required for Perch embedding extraction.")

perch_model_dir = find_perch_model_dir()
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


def explain_perch_runtime_error(error: Exception) -> None:
    message = str(error)
    if "XlaCallModuleOp with version 10 is not supported" in message:
        raise RuntimeError(
            "The attached Perch SavedModel requires a newer TensorFlow/XLA runtime than this Kaggle image. "
            "The setup cell should install tensorflow>=2.20 before TensorFlow is imported. Restart the "
            "Kaggle session after installation, then run the notebook from the top. If internet is disabled, "
            "attach a Kaggle dataset containing a compatible TensorFlow wheel or use an older Perch SavedModel export."
        ) from error
    raise error


def smoke_test_perch() -> None:
    dummy = tf.zeros((1, int(CFG.sample_rate * CFG.duration)), dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: dummy})
        else:
            outputs = infer(dummy)
    except Exception as error:
        explain_perch_runtime_error(error)
    print({name: tuple(value.shape) for name, value in outputs.items()})


smoke_test_perch()


I0000 00:00:1779448169.810888      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779448169.813880      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Perch model: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
Inputs: ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
Outputs: {'embedding': TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding'), 'spatial_embedding': TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding'), 'spectrogram': TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram'), 'label': TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')}


2026-05-22 11:09:39.675529: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:09:39.818129: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:09:41.146955: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:09:41.286851: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:09:41.474632: E external/local_xla/xla/stream_

{'embedding': (1, 1536), 'spatial_embedding': (1, 16, 4, 1536), 'spectrogram': (1, 500, 128), 'label': (1, 14795)}


## 4. Embedding Extraction


Convert each 5-second waveform into a Perch embedding and cache the matrix. Once cached, the probe can be retrained without rerunning TensorFlow inference.


In [4]:
def load_audio(path: Path) -> np.ndarray:
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    if keyword_specs:
        input_name = next(iter(keyword_specs))
        try:
            outputs = infer(**{input_name: tensor})
        except Exception as error:
            explain_perch_runtime_error(error)
    else:
        try:
            outputs = infer(tensor)
        except Exception as error:
            explain_perch_runtime_error(error)
    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (name for name in arrays if "embedding" in name.lower() or "embed" in name.lower()),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


embeddings_path = CFG.artifact_dir / "train_embeddings.npz"
if embeddings_path.exists():
    saved = np.load(embeddings_path, allow_pickle=True)
    embeddings = saved["embeddings"]
else:
    chunks = []
    waveforms = []
    for path in tqdm(train["filepath"], desc="audio"):
        waveforms.append(load_audio(path))
        if len(waveforms) == CFG.extraction_batch_size:
            chunks.append(run_perch_batch(np.stack(waveforms)))
            waveforms = []
    if waveforms:
        chunks.append(run_perch_batch(np.stack(waveforms)))
    embeddings = np.concatenate(chunks, axis=0)
    np.savez_compressed(
        embeddings_path,
        embeddings=embeddings,
        labels=train["primary_label"].to_numpy(),
        filenames=train["filename"].to_numpy(),
    )

print(f"Embeddings: {embeddings.shape}")
print(f"Saved: {embeddings_path}")


audio:   0%|          | 0/35549 [00:00<?, ?it/s]

2026-05-22 11:10:06.709576: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:10:06.861396: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:10:08.230597: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:10:08.374579: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 11:10:08.575565: E external/local_xla/xla/stream_

Embeddings: (35549, 1536)
Saved: /kaggle/working/artifacts/perch_v2/train_embeddings.npz


## 5. Probe Model


The probe is deliberately shallow because Perch should already encode strong acoustic structure. The classifier learns a compact boundary on top of frozen features.


In [5]:
class PerchProbe(nn.Module):
    def __init__(self, embedding_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


x = embeddings.astype(np.float32)
y = train["target"].to_numpy(dtype=np.int64)


def safe_train_valid_split(targets: np.ndarray, test_size: float = 0.2) -> tuple[np.ndarray, np.ndarray]:
    counts = pd.Series(targets).value_counts()
    rare_classes = set(counts[counts < 2].index)
    all_idx = np.arange(len(targets))
    rare_idx = np.array([idx for idx in all_idx if targets[idx] in rare_classes], dtype=np.int64)
    common_idx = np.array([idx for idx in all_idx if targets[idx] not in rare_classes], dtype=np.int64)

    train_common, valid_idx = train_test_split(
        common_idx,
        test_size=test_size,
        random_state=CFG.seed,
        stratify=targets[common_idx],
    )
    train_idx = np.concatenate([train_common, rare_idx])
    return train_idx, valid_idx


train_idx, valid_idx = safe_train_valid_split(y)
print(f"Probe train rows: {len(train_idx):,}")
print(f"Probe valid rows: {len(valid_idx):,}")
print(f"Classes with fewer than 2 rows kept in train only: {(pd.Series(y).value_counts() < 2).sum():,}")

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
    batch_size=CFG.probe_batch_size,
    shuffle=True,
)
valid_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
    batch_size=CFG.probe_batch_size * 2,
    shuffle=False,
)

model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)


Probe train rows: 28,440
Probe valid rows: 7,109
Classes with fewer than 2 rows kept in train only: 4


## 6. Training


Rare singleton classes remain in the training split to keep stratification valid. Validation accuracy is a representation-quality signal, not a complete proxy for hidden soundscape score.


In [6]:
@torch.no_grad()
def validate() -> float:
    model.eval()
    correct = 0
    seen = 0
    for xb, yb in valid_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += xb.size(0)
    return correct / max(seen, 1)


history = []
best_acc = 0.0
epochs_without_improvement = 0

for epoch in range(1, CFG.probe_epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    valid_acc = validate()
    row = {
        "epoch": epoch,
        "train_loss": total_loss / max(len(train_loader.dataset), 1),
        "valid_acc": valid_acc,
    }
    history.append(row)
    print(row)

    improved = valid_acc > best_acc + CFG.early_stopping_min_delta
    if improved:
        best_acc = valid_acc
        epochs_without_improvement = 0
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_perch_probe.pt",
        )
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.early_stopping_patience:
            print(f"Early stopping after {epoch} epochs; best valid accuracy: {best_acc:.4f}")
            break

pd.DataFrame(history).to_csv(CFG.artifact_dir / "history.csv", index=False)
print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Saved outputs to {CFG.artifact_dir}")


{'epoch': 1, 'train_loss': 1.307341811023181, 'valid_acc': 0.8357012238008159}
{'epoch': 2, 'train_loss': 0.5470918192684063, 'valid_acc': 0.8333098888732593}
{'epoch': 3, 'train_loss': 0.3381599192797048, 'valid_acc': 0.8392178928119285}
{'epoch': 4, 'train_loss': 0.2108916793107651, 'valid_acc': 0.83738922492615}
{'epoch': 5, 'train_loss': 0.14286931101782915, 'valid_acc': 0.8335912223941483}
{'epoch': 6, 'train_loss': 0.11054961122205656, 'valid_acc': 0.8331692221128147}
{'epoch': 7, 'train_loss': 0.09132488072861598, 'valid_acc': 0.8366858911239274}
Early stopping after 7 epochs; best valid accuracy: 0.8392
Best valid accuracy: 0.8392
Saved outputs to /kaggle/working/artifacts/perch_v2


## 7. Interpretation

- Perch embeddings provide the strongest validation signal in this repo and are best used as teacher features or distillation targets.
- Direct Perch inference is operationally heavier than EfficientNet, so keep it experimental until hidden-test runtime is proven.
- The embedding cache is a Kaggle artifact and should stay outside Git.


## 8. Artifacts


Package the embedding cache, probe checkpoint, label map, and training history for Kaggle download. The embedding file is large and remains outside Git.


In [7]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))


Wrote /kaggle/working/birdclef_perch_v2_artifacts.zip (195.95 MB)


/kaggle/working/birdclef_perch_v2_artifacts.zip